# Figure 6 New New

Pilot notebook for the full-nucleotide ParD3 D-F pipeline. The raw-derived `G_mu` products are generated externally by `scripts/generate_figure6_full_nuc_gmu.py`; this notebook loads those compact products, fits `rho_2^fit`, and plots mean plus standard deviation across 20 deterministic random orderings.

In [ ]:
from __future__ import annotations

from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np

from slide.direvo_functions import get_single_decay_rate_IK_v2, model_function_IK_v2
from slide.utils import (
    FIGURE_LABEL_SIZE,
    FIGURE_LEGEND_SIZE,
    FIGURE_TICK_SIZE,
    FIGURE_TITLE_SIZE,
    PANEL_LETTER_SIZE,
    get_figures_dir,
    get_processed_data_dir,
    get_raw_data_dir,
    load_pickle,
    save_pickle,
)

OVERWRITE_RAW_PKL: bool = False
OVERWRITE_PROCESSED_PKL: bool = False
PLOT_ONLY: bool = True
SAVE_FIGURES: bool = True
SAVE_TYPE_LIST = ("pdf", "png", "eps")
PANEL_DPI = 350

RAW_DATA_DIR = get_raw_data_dir()
PROCESSED_DATA_DIR = get_processed_data_dir()
FIGURES_DIR = get_figures_dir()
for figure_type in SAVE_TYPE_LIST:
    (FIGURES_DIR / figure_type).mkdir(parents=True, exist_ok=True)

DF_TOTAL_MUTATION_RATE: float = 0.1
DF_NUM_GENERATIONS: int = 75
FIT_FIXED_AMPLITUDE: bool = False
PILOT_LANDSCAPE: str = "pard3"
PROCESSED_PATH = PROCESSED_DATA_DIR / "figure6_full_nuc_gmu_pard3_processed.pkl"

MODEL_KEYS = (
    "nuc_uniform",
    "nuc_e_coli_directed",
    "nuc_a_thaliana_directed",
    "nuc_human_directed",
)
MODEL_LABELS = {
    "nuc_uniform": "Uniform",
    "nuc_e_coli_directed": "E. coli",
    "nuc_a_thaliana_directed": "A. thaliana",
    "nuc_human_directed": "Human",
}
MODEL_COLORS = {
    "nuc_uniform": "#4c78a8",
    "nuc_e_coli_directed": "#f58518",
    "nuc_a_thaliana_directed": "#54a24b",
    "nuc_human_directed": "#b279a2",
}

print(f"raw_data: {RAW_DATA_DIR}")
print(f"processed_data: {PROCESSED_DATA_DIR}")
print(f"figures: {FIGURES_DIR}")


def save_panel_figure(fig: plt.Figure, stem: str, *, bbox_inches: str = "tight") -> None:
    """Save a figure in each configured output format.

    Parameters:
    - fig: plt.Figure
        Figure to save.
    - stem: str
        Output filename stem.
    - bbox_inches: str
        Matplotlib bounding-box option.

    Returns:
    - None
        Writes figure files when ``SAVE_FIGURES`` is true.
    """
    if not SAVE_FIGURES:
        return
    for figure_type in SAVE_TYPE_LIST:
        fig.savefig(FIGURES_DIR / figure_type / f"{stem}.{figure_type}", dpi=PANEL_DPI, bbox_inches=bbox_inches)


def add_panel_letter(ax: plt.Axes, letter: str) -> None:
    """Add a panel letter to an axis.

    Parameters:
    - ax: plt.Axes
        Axis to annotate.
    - letter: str
        Panel letter.

    Returns:
    - None
        Mutates ``ax`` in place.
    """
    ax.text(-0.16, 1.08, letter, transform=ax.transAxes, fontsize=PANEL_LETTER_SIZE, fontweight="bold", va="top")


## Raw Products

When `PLOT_ONLY = False`, this notebook generates missing raw products by calling `scripts/generate_figure6_full_nuc_gmu.py`. With `PLOT_ONLY = True`, missing raw products raise an error rather than accidentally launching a long simulation.

```bash
conda run -n SLIDE_env python scripts/generate_figure6_full_nuc_gmu.py --landscapes pard3 --models nuc_uniform,nuc_e_coli_directed,nuc_a_thaliana_directed,nuc_human_directed
```

In [ ]:
RAW_FILES = {
    model: f"figure6_full_nuc_gmu_{PILOT_LANDSCAPE}_{model}_raw.pkl"
    for model in MODEL_KEYS
}


def raw_path(model: str) -> Path:
    """Return the raw-derived full-nucleotide G_mu path for a model.

    Parameters:
    - model: str
        Mutation model key.

    Returns:
    - pathlib.Path
        Path in ``raw_data``.
    """
    return RAW_DATA_DIR / RAW_FILES[model]


def get_missing_raw_paths() -> list[Path]:
    """Return missing ParD3 full-nucleotide G_mu payload paths.

    Parameters:
    - None

    Returns:
    - list[pathlib.Path]
        Missing raw-derived payload paths.
    """
    return [raw_path(model) for model in MODEL_KEYS if not raw_path(model).exists()]


def generate_missing_raw_products() -> None:
    """Generate missing ParD3 full-nucleotide G_mu payloads.

    Parameters:
    - None

    Returns:
    - None
        Writes raw-derived payloads into ``RAW_DATA_DIR``.
    """
    command = [
        sys.executable,
        str(Path("scripts/generate_figure6_full_nuc_gmu.py")),
        "--landscapes",
        PILOT_LANDSCAPE,
        "--models",
        ",".join(MODEL_KEYS),
        "--output-dir",
        str(RAW_DATA_DIR),
    ]
    if OVERWRITE_RAW_PKL:
        command.append("--overwrite")
    subprocess.run(command, check=True)


missing_raw_paths = get_missing_raw_paths()
print(f"Missing ParD3 full-nucleotide D-F raw products: {len(missing_raw_paths)}")
for path in missing_raw_paths:
    print(f"  {path}")


## Processed D-F Fits

In [ ]:
def load_full_nuc_payloads() -> dict[str, dict[str, object]]:
    """Load all ParD3 full-nucleotide G_mu payloads.

    Parameters:
    - None

    Returns:
    - dict[str, dict[str, object]]
        Payloads keyed by mutation model.
    """
    current_missing_raw_paths = get_missing_raw_paths()
    if current_missing_raw_paths and PLOT_ONLY:
        listing = "\n".join(str(path) for path in current_missing_raw_paths)
        raise FileNotFoundError(
            "Missing ParD3 full-nucleotide D-F raw products and PLOT_ONLY is True. "
            "Set PLOT_ONLY = False to generate them from this notebook.\n"
            f"{listing}"
        )
    if current_missing_raw_paths or (OVERWRITE_RAW_PKL and not PLOT_ONLY):
        generate_missing_raw_products()
    current_missing_raw_paths = get_missing_raw_paths()
    if current_missing_raw_paths:
        listing = "\n".join(str(path) for path in current_missing_raw_paths)
        raise FileNotFoundError(f"Missing ParD3 full-nucleotide D-F raw products after generation:\n{listing}")
    return {model: load_pickle(raw_path(model)) for model in MODEL_KEYS}


def normalise_curve(curve: np.ndarray) -> np.ndarray:
    """Normalise a G_mu curve by its first generation value.

    Parameters:
    - curve: np.ndarray
        One-dimensional G_mu curve.

    Returns:
    - np.ndarray
        Normalised curve with finite values.
    """
    curve = np.asarray(curve, dtype=float)
    denominator = max(float(curve[0]), 1e-10)
    return curve / denominator


def fit_rho2_curve(curve: np.ndarray) -> tuple[float, float, float, np.ndarray]:
    """Fit a normalised G_mu decay curve.

    Parameters:
    - curve: np.ndarray
        One-dimensional unnormalised G_mu curve.

    Returns:
    - tuple[float, float, float, np.ndarray]
        Fitted rho_2, amplitude, asymptote, and fitted normalised curve.
    """
    normalised = normalise_curve(curve)
    rho_raw, amplitude, asymptote = get_single_decay_rate_IK_v2(
        normalised,
        mut=DF_TOTAL_MUTATION_RATE,
        num_steps=DF_NUM_GENERATIONS,
        fix_amplitude=FIT_FIXED_AMPLITUDE,
    )
    steps = np.arange(DF_NUM_GENERATIONS)
    fitted = model_function_IK_v2(
        steps,
        rho_raw,
        amplitude,
        asymptote,
        mut=DF_TOTAL_MUTATION_RATE,
        fix_amplitude=FIT_FIXED_AMPLITUDE,
        F0=float(normalised[0]),
    )
    return float(rho_raw / 2.0), float(amplitude), float(asymptote), np.asarray(fitted, dtype=float)


def process_full_nuc_payloads(raw_by_model: dict[str, dict[str, object]]) -> dict[str, object]:
    """Fit rho_2 for every model, ordering, and start count.

    Parameters:
    - raw_by_model: dict[str, dict[str, object]]
        Full-nucleotide G_mu payloads keyed by mutation model.

    Returns:
    - dict[str, object]
        Processed fit summaries and final-count decay curves.
    """
    processed: dict[str, object] = {}
    num_fit_failures = 0
    for model, payload in raw_by_model.items():
        data = payload["data"]
        g_mu = np.asarray(data["g_mu"], dtype=float)
        counts = np.asarray(data["counts"], dtype=int)
        included_counts = np.asarray(data["included_counts"], dtype=int)
        if g_mu.shape != (20, len(counts), DF_NUM_GENERATIONS):
            raise ValueError(f"Unexpected g_mu shape for {model}: {g_mu.shape}")
        if not np.array_equal(included_counts, np.broadcast_to(counts[None, :], included_counts.shape)):
            raise ValueError(f"Included counts do not match counts for {model}.")
        if not np.isfinite(g_mu).all():
            raise ValueError(f"Non-finite g_mu values for {model}.")

        rho2 = np.full(g_mu.shape[:2], np.nan, dtype=float)
        amplitude = np.full(g_mu.shape[:2], np.nan, dtype=float)
        asymptote = np.full(g_mu.shape[:2], np.nan, dtype=float)
        fitted_curves = np.full(g_mu.shape, np.nan, dtype=float)
        for index in np.ndindex(g_mu.shape[:2]):
            try:
                rho2[index], amplitude[index], asymptote[index], fitted_curves[index] = fit_rho2_curve(g_mu[index])
            except RuntimeError:
                num_fit_failures += 1

        final_curves = np.asarray([normalise_curve(curve) for curve in g_mu[:, -1, :]], dtype=float)
        processed[model] = {
            "counts": counts,
            "rho2": rho2,
            "rho2_mean": np.nanmean(rho2, axis=0),
            "rho2_std": np.nanstd(rho2, axis=0, ddof=1),
            "amplitude": amplitude,
            "asymptote": asymptote,
            "fitted_curves": fitted_curves,
            "final_observed_mean": final_curves.mean(axis=0),
            "final_observed_std": final_curves.std(axis=0, ddof=1),
            "final_fitted_mean": np.nanmean(fitted_curves[:, -1, :], axis=0),
        }

    return {
        "data": processed,
        "params": {
            "landscape": PILOT_LANDSCAPE,
            "models": MODEL_KEYS,
            "num_steps": DF_NUM_GENERATIONS,
            "total_mutation_rate": DF_TOTAL_MUTATION_RATE,
            "fit_fixed_amplitude": FIT_FIXED_AMPLITUDE,
        },
        "metadata": {
            "paper_reference": "Figure 6D-F pilot",
            "description": "ParD3 full-nucleotide D-F fits from compact G_mu products.",
            "num_fit_failures": num_fit_failures,
            "standard_deviation_ddof": 1,
        },
    }


if PROCESSED_PATH.exists() and (PLOT_ONLY or not OVERWRITE_PROCESSED_PKL):
    figure6_full_nuc_processed = load_pickle(PROCESSED_PATH)
else:
    full_nuc_raw_by_model = load_full_nuc_payloads()
    figure6_full_nuc_processed = process_full_nuc_payloads(full_nuc_raw_by_model)
    save_pickle(figure6_full_nuc_processed, PROCESSED_PATH)

print(f"Fit failures: {figure6_full_nuc_processed['metadata']['num_fit_failures']}")


## Alternative D-F Panels

In [ ]:
def plot_pard3_full_nuc_sampling(ax: plt.Axes) -> None:
    """Plot rho_2 fit versus number of full-nucleotide starts for ParD3.

    Parameters:
    - ax: plt.Axes
        Axis to draw on.

    Returns:
    - None
        Mutates ``ax`` in place.
    """
    processed = figure6_full_nuc_processed["data"]
    for model in MODEL_KEYS:
        model_data = processed[model]
        counts = np.asarray(model_data["counts"], dtype=float)
        mean = np.asarray(model_data["rho2_mean"], dtype=float)
        std = np.asarray(model_data["rho2_std"], dtype=float)
        color = MODEL_COLORS[model]
        ax.plot(counts, mean, marker="o", ms=3, lw=1.2, color=color, label=MODEL_LABELS[model])
        ax.fill_between(counts, mean - std, mean + std, color=color, alpha=0.18, lw=0)
    ax.set_xscale("log")
    ax.set_xlabel("Number of nucleotide starts", fontsize=FIGURE_LABEL_SIZE)
    ax.set_ylabel(r"Fitted $\rho_2$", fontsize=FIGURE_LABEL_SIZE)
    ax.set_title("ParD3 full nucleotide starts", fontsize=FIGURE_TITLE_SIZE)
    ax.tick_params(labelsize=FIGURE_TICK_SIZE)
    ax.legend(frameon=False, fontsize=FIGURE_LEGEND_SIZE)


fig, ax = plt.subplots(figsize=(3.4, 2.6))
add_panel_letter(ax, "D")
plot_pard3_full_nuc_sampling(ax)
fig.tight_layout()
save_panel_figure(fig, "figure_6_new_new_pard3_full_nuc_df")
plt.show()


## Final-Count Decay Curves

In [ ]:
def plot_final_count_decay_curves(ax: plt.Axes) -> None:
    """Plot final-count observed and fitted decay curves for all models.

    Parameters:
    - ax: plt.Axes
        Axis to draw on.

    Returns:
    - None
        Mutates ``ax`` in place.
    """
    steps = np.arange(DF_NUM_GENERATIONS)
    processed = figure6_full_nuc_processed["data"]
    for model in MODEL_KEYS:
        model_data = processed[model]
        color = MODEL_COLORS[model]
        observed = np.asarray(model_data["final_observed_mean"], dtype=float)
        fitted = np.asarray(model_data["final_fitted_mean"], dtype=float)
        ax.plot(steps, observed, color=color, lw=1.2, label=MODEL_LABELS[model])
        ax.plot(steps, fitted, color=color, lw=1.0, ls="--", alpha=0.9)
    ax.set_xlabel("Generation", fontsize=FIGURE_LABEL_SIZE)
    ax.set_ylabel(r"Normalised $G_\mu$", fontsize=FIGURE_LABEL_SIZE)
    ax.set_title("Final full-space prefix", fontsize=FIGURE_TITLE_SIZE)
    ax.tick_params(labelsize=FIGURE_TICK_SIZE)
    ax.legend(frameon=False, fontsize=FIGURE_LEGEND_SIZE)


fig, ax = plt.subplots(figsize=(3.4, 2.6))
plot_final_count_decay_curves(ax)
fig.tight_layout()
save_panel_figure(fig, "figure_6_new_new_pard3_full_nuc_decay_curves")
plt.show()
